# Lab 3 - 09
Codificamos **Sabor**, **Tamaño** y **Hielo** en columnas binarias y modelamos
la **Calificacion** con OLS.

In [1]:
import pandas as pd          # manejo del DataFrame
import statsmodels.api as sm  # regresion OLS con tabla de resultados

# Leemos la hoja de Excel. Cada fila es una prueba: un sabor, un tamaño,
# con o sin hielo, y la calificacion que le dieron.
df = pd.read_excel('Libro1.xlsx')
df

,Sabor,Tamaño,Hielo,Calificacion
0,Naranja 1,Ch,Si,8.5
1,Naranja 2,Ch,No,7.0
2,Naranja 3,G,Si,9.0
3,Naranja 4,G,No,7.5
4,Coco 1,Ch,Si,7.5
5,Coco 2,Ch,No,6.0
6,Coco 3,G,Si,8.0
7,Coco 4,G,No,7.0
8,Kahlua 1,Ch,Si,9.0
9,Kahlua 2,Ch,No,7.0


## 1. Dummies del sabor con `str.contains`
La regresion solo entiende numeros, no texto como "Naranja". Por eso convertimos
el sabor en 5 columnas: **1** si la fila es de ese sabor, **0** si no.

In [2]:
sabores = ['Naranja', 'Coco', 'Kahlua', 'Fresa-Piña', 'Sandia']

for sabor in sabores:
    # str.contains busca el nombre del sabor dentro del texto de la columna.
    # Sirve aunque venga con el numero de repeticion pegado ('Naranja 1', 'Naranja 2'...).
    # Devuelve True/False, y astype(int) lo convierte a 1/0.
    df[sabor] = df['Sabor'].str.contains(sabor).astype(int)

df

,Sabor,Tamaño,Hielo,Calificacion,Naranja,Coco,Kahlua,Fresa-Piña,Sandia
0,Naranja 1,Ch,Si,8.5,1,0,0,0,0
1,Naranja 2,Ch,No,7.0,1,0,0,0,0
2,Naranja 3,G,Si,9.0,1,0,0,0,0
3,Naranja 4,G,No,7.5,1,0,0,0,0
4,Coco 1,Ch,Si,7.5,0,1,0,0,0
5,Coco 2,Ch,No,6.0,0,1,0,0,0
6,Coco 3,G,Si,8.0,0,1,0,0,0
7,Coco 4,G,No,7.0,0,1,0,0,0
8,Kahlua 1,Ch,Si,9.0,0,0,1,0,0
9,Kahlua 2,Ch,No,7.0,0,0,1,0,0


## 2. Dummies de Tamaño y Hielo
Estas dos solo tienen dos niveles (Ch/G y Si/No), asi que basta **una** columna
para cada una. La categoria que dejamos fuera queda como referencia:

- `Tamaño_G` = 1 si es Grande, 0 si es Chico → el Chico es la referencia
- `Hielo_Si` = 1 si lleva hielo, 0 si no → el No es la referencia

Si pusieramos las dos columnas de cada variable (Ch y G) volveriamos a tener
colinealidad, porque una seria el opuesto exacto de la otra.

In [3]:
# La comparacion == devuelve True/False y astype(int) lo pasa a 1/0
df['Tamaño_G'] = (df['Tamaño'] == 'G').astype(int)
df['Hielo_Si'] = (df['Hielo'] == 'Si').astype(int)

df[['Sabor', 'Tamaño', 'Tamaño_G', 'Hielo', 'Hielo_Si']]

,Sabor,Tamaño,Tamaño_G,Hielo,Hielo_Si
0,Naranja 1,Ch,0,Si,1
1,Naranja 2,Ch,0,No,0
2,Naranja 3,G,1,Si,1
3,Naranja 4,G,1,No,0
4,Coco 1,Ch,0,Si,1
5,Coco 2,Ch,0,No,0
6,Coco 3,G,1,Si,1
7,Coco 4,G,1,No,0
8,Kahlua 1,Ch,0,Si,1
9,Kahlua 2,Ch,0,No,0


## 4. Modelo completo: Sabor + Tamaño + Hielo
Agregamos las dos columnas nuevas a la matriz X. Los betas cambian de interpretacion:

- los de **sabor** son la calificacion base de ese sabor (en Chico y sin hielo)
- `Tamaño_G` es cuanto **sube o baja** la calificacion al pasar de Chico a Grande
- `Hielo_Si` es cuanto cambia al agregarle hielo

Como el diseño esta balanceado (cada sabor se probo en las 4 combinaciones),
estos dos efectos son iguales para todos los sabores.

In [5]:
# Ahora X lleva las 5 dummies de sabor mas las 2 columnas nuevas
X_full = df[sabores + ['Tamaño_G', 'Hielo_Si']]

modelo = sm.OLS(y, X_full).fit()

print('R2:', modelo.rsquared)
modelo.params

R2: 0.9220338983050848


Naranja       6.850
Coco          5.975
Kahlua        7.225
Fresa-Piña    6.475
Sandia        5.225
Tamaño_G      0.500
Hielo_Si      1.800
dtype: float64

In [7]:
# De donde salen esos dos betas: son la diferencia de promedios entre niveles
print('Efecto del tamaño:', df.groupby('Tamaño')['Calificacion'].mean().diff()['G'])
print('Efecto del hielo  :', df.groupby('Hielo')['Calificacion'].mean().diff()['Si'])

Efecto del tamaño: 0.5
Efecto del hielo  : 1.8000000000000007


## 5. Lo mismo con `OneHotEncoder`
En vez de armar las dummies a mano, dejamos que sklearn las construya.
Es lo que se usa en la practica: el encoder aprende las categorias con `fit`
y luego transforma datos nuevos con el mismo criterio.

In [8]:
from sklearn.preprocessing import OneHotEncoder

# El encoder trata cada texto distinto como una categoria, asi que 'Naranja 1' y
# 'Naranja 2' serian dos categorias diferentes. Nos quedamos solo con la primera
# palabra para tener el sabor limpio.
df['Sabor_base'] = df['Sabor'].str.split().str[0]

# sparse_output=False para que regrese un arreglo normal y no una matriz dispersa;
# dtype=int para ver 1/0 en vez de 1.0/0.0
enc_sabor = OneHotEncoder(sparse_output=False, dtype=int)

# drop='if_binary' deja una sola columna en las variables de dos niveles,
# que es justo lo que hicimos a mano con Tamaño_G y Hielo_Si
enc_binarias = OneHotEncoder(sparse_output=False, dtype=int, drop='if_binary')

# fit_transform hace dos cosas: aprende las categorias (fit) y genera la matriz
# de 1 y 0 (transform). Va con doble corchete porque espera 2 dimensiones.
# get_feature_names_out() nos da los nombres de las columnas que creo.
X_sabor = pd.DataFrame(enc_sabor.fit_transform(df[['Sabor_base']]),
                       columns=enc_sabor.categories_[0])

X_bin = pd.DataFrame(enc_binarias.fit_transform(df[['Tamaño', 'Hielo']]),
                     columns=enc_binarias.get_feature_names_out())

# Pegamos los dos bloques uno al lado del otro (axis=1 = por columnas)
X_ohe = pd.concat([X_sabor, X_bin], axis=1)
X_ohe

,Coco,Fresa-Piña,Kahlua,Naranja,Sandia,Tamaño_G,Hielo_Si
0,0,0,0,1,0,0,1
1,0,0,0,1,0,0,0
2,0,0,0,1,0,1,1
3,0,0,0,1,0,1,0
4,1,0,0,0,0,0,1
5,1,0,0,0,0,0,0
6,1,0,0,0,0,1,1
7,1,0,0,0,0,1,0
8,0,0,1,0,0,0,1
9,0,0,1,0,0,0,0


In [9]:
# Misma regresion, pero con las columnas que genero el encoder
modelo_ohe = sm.OLS(y, X_ohe).fit()

print('R2:', modelo_ohe.rsquared)
modelo_ohe.params

R2: 0.9220338983050848


Coco          5.975
Fresa-Piña    6.475
Kahlua        7.225
Naranja       6.850
Sandia        5.225
Tamaño_G      0.500
Hielo_Si      1.800
dtype: float64

Mismos betas que en el paso 4: solo cambia el orden, porque el encoder
ordena las categorias alfabeticamente.